# KDD Stage 1 — Selection

**Real data used in this run:** Open-Meteo historical reanalysis for Durban (KZN-ETHEKWINI)
and Mbombela (MP-EHLANZENI), 2021-06-01 to 2022-06-30, supplied by the user as
`data/raw/source_downloads/*.xlsx`. These files are Open-Meteo CSV exports saved as Excel
workbooks (one text column per row, not a normal spreadsheet) — parsed accordingly below.
See `data/raw/source_downloads/Dataset_README.md` for provenance and the exact download URL.

**Everything else is still synthetic**, generated with the same approach as before, because
no network access was available to fetch satellite imagery, SANLC land cover, river-level/
hydrological data, or a full EM-DAT export while building this prototype:
- 3 additional regions (`KZN-UMGUNGUNDLOVU`, `KZN-KINGCETSHWAYO`, `MP-NKANGALA`) have no real
  data supplied, so they are fully synthetic, tagged `data_source=synthetic_supplementary`.
- River level and soil moisture are **not present in the real data at all** (Open-Meteo gives
  rain/temperature/humidity/pressure/wind only) — they are modelled proxies derived from the
  antecedent precipitation index for every region, real and synthetic alike. This is a real
  limitation: it means the model never sees a true measured river level.
- Satellite tiles and SANLC land-cover composition remain synthetic placeholders for all 5
  regions (no imagery pipeline exists in this prototype yet).
- Pressure and wind ARE present in the real data and are kept in `station_readings.csv` for
  completeness/future use, but are **not** fed into the trained model in notebooks 03-04, so
  the trained artefact stays consistent with what `backend/app/ml/predictor.py` expects
  (rainfall, river_level, soil_moisture, temperature, humidity, api_index). Wiring pressure/
  wind into the model is a follow-up, not done here.

**Flood labels for the real-data regions are a heuristic, not verified EM-DAT ground truth**
(see the labelling cell below) — except for 2022-04-11 and 2022-05-21/22, which are the
widely-reported April/May 2022 KZN flood dates and are asserted as label=1 regardless of the
threshold rule, as a sanity anchor.

In [ ]:
import numpy as np
import pandas as pd
import hashlib
import json
import io
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

REGIONS = [
    ("KZN-ETHEKWINI", "eThekwini (Durban)", "KwaZulu-Natal"),
    ("KZN-UMGUNGUNDLOVU", "uMgungundlovu", "KwaZulu-Natal"),
    ("KZN-KINGCETSHWAYO", "King Cetshwayo", "KwaZulu-Natal"),
    ("MP-EHLANZENI", "Ehlanzeni", "Mpumalanga"),
    ("MP-NKANGALA", "Nkangala", "Mpumalanga"),
]
REAL_REGIONS = {"KZN-ETHEKWINI": "durban_kzn_daily_2021-2022.xlsx",
                "MP-EHLANZENI": "mbombela_mpumalanga_daily_2021-2022_csv.xlsx"}
KNOWN_FLOOD_DATES = {"KZN-ETHEKWINI": ["2022-04-11", "2022-04-12", "2022-05-21", "2022-05-22"]}
# ^ widely reported April & May 2022 KZN flood dates (public reporting, not a formal EM-DAT
# export we had access to). Used only to force label=1 on the two confirmed real events.

DATASET_VERSION = "real-openmeteo-v1+synthetic-supplement"
SRC = Path("../data/raw/source_downloads")
RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)
TILE = 16
N_DAYS_SYNTH = 420  # only used for the 3 fully-synthetic regions

In [ ]:
def load_openmeteo_xlsx(path: Path) -> pd.DataFrame:
    """Open-Meteo archive CSV export saved as .xlsx: row0=metadata, row1=lat/lon, row2=blank,
    row3=header, row4+=data — each stored as one text cell, not real spreadsheet columns."""
    raw = pd.read_excel(path, header=None)
    lat, lon = (float(x) for x in raw.iloc[1, 0].split(",")[:2])
    header_line = raw.iloc[3, 0]
    data_lines = raw.iloc[4:, 0].dropna().tolist()
    csv_text = header_line + "\n" + "\n".join(data_lines)
    df = pd.read_csv(io.StringIO(csv_text))
    # source header names are mangled by the degree symbol's encoding; rename positionally
    # instead of trusting the raw header text
    df.columns = ["date", "rainfall_mm_24h", "temperature_c", "humidity_pct",
                  "pressure_hpa", "wind_speed_kmh"]
    df["date"] = pd.to_datetime(df["date"])
    df["centroid_lat"], df["centroid_lon"] = lat, lon
    return df

In [ ]:
def derive_proxies_and_labels(df: pd.DataFrame, region_id: str) -> pd.DataFrame:
    """River level and soil moisture are NOT in the real data. Both are modelled proxies from
    the antecedent precipitation index (API, decay 0.9) — the same construction used for the
    fully-synthetic regions, so the two data types are at least internally consistent even
    though these two columns are never truly 'measured' for the real regions."""
    df = df.sort_values("date").reset_index(drop=True)
    api = np.zeros(len(df))
    for i in range(1, len(df)):
        api[i] = 0.9 * api[i - 1] + df["rainfall_mm_24h"].iloc[i]
    df["river_level_m"] = (0.6 + 0.02 * api).round(2)
    df["soil_moisture"] = np.clip(0.25 + api / 400, 0, 1).round(3)

    r3 = df["rainfall_mm_24h"].rolling(3, min_periods=1).sum()
    # Heuristic heavy-rainfall threshold (not an official SAWS warning-level lookup — those
    # vary by district/season and we did not have access to the SAWS warning-level table).
    # 50mm/24h and 80mm/3-day are commonly cited heavy-rain reference points for this region.
    heuristic = ((df["rainfall_mm_24h"] >= 50) | (r3 >= 80)).astype(int)

    known = KNOWN_FLOOD_DATES.get(region_id, [])
    forced = df["date"].dt.strftime("%Y-%m-%d").isin(known).astype(int)
    df["flood_label"] = np.maximum(heuristic.values, forced.values)
    df["region_id"] = region_id
    df["data_source"] = "real_openmeteo"
    return df

In [ ]:
real_frames = []
for region_id, fname in REAL_REGIONS.items():
    df = load_openmeteo_xlsx(SRC / fname)
    df = derive_proxies_and_labels(df, region_id)
    real_frames.append(df)
    print(region_id, "rows:", len(df), "positive days:", int(df["flood_label"].sum()))

pd.concat(real_frames)[["region_id", "date", "rainfall_mm_24h", "flood_label"]].query("flood_label == 1")

In [ ]:
def simulate_synthetic_region(region_id: str, seed: int) -> pd.DataFrame:
    """Unchanged synthetic generator from the previous iteration, for the 3 regions with no
    real data supplied. Seasonal baseline + noise + injected flood episodes."""
    r = np.random.default_rng(seed)
    days = pd.date_range("2024-01-01", periods=N_DAYS_SYNTH, freq="D")
    doy = days.dayofyear.values
    season = 40 + 35 * np.sin(2 * np.pi * (doy - 60) / 365)
    rainfall = np.clip(season * 0.4 + r.normal(0, 12, N_DAYS_SYNTH), 0, None)

    label = np.zeros(N_DAYS_SYNTH, dtype=int)
    n_events = r.integers(4, 8)
    for _ in range(n_events):
        start = r.integers(10, N_DAYS_SYNTH - 15)
        dur = r.integers(3, 8)
        rainfall[start:start + dur] += r.uniform(80, 220, dur)
        label[start:start + dur] = 1

    api = np.zeros(N_DAYS_SYNTH)
    for i in range(1, N_DAYS_SYNTH):
        api[i] = 0.9 * api[i - 1] + rainfall[i]
    river_level = np.clip(0.6 + 0.02 * api + r.normal(0, 0.15, N_DAYS_SYNTH), 0, None)
    soil_moisture = np.clip(0.25 + api / 400 + r.normal(0, 0.03, N_DAYS_SYNTH), 0, 1)
    temperature = 22 - 6 * np.cos(2 * np.pi * doy / 365) + r.normal(0, 2, N_DAYS_SYNTH)
    humidity = np.clip(60 + 20 * (rainfall > 20) + r.normal(0, 8, N_DAYS_SYNTH), 10, 100)
    pressure = 1018 + r.normal(0, 6, N_DAYS_SYNTH) - (rainfall > 40) * 8
    wind = np.clip(15 + r.normal(0, 8, N_DAYS_SYNTH) + (rainfall > 40) * 10, 0, None)

    df = pd.DataFrame({
        "region_id": region_id, "date": days, "rainfall_mm_24h": rainfall.round(1),
        "river_level_m": river_level.round(2), "soil_moisture": soil_moisture.round(3),
        "temperature_c": temperature.round(1), "humidity_pct": humidity.round(1),
        "pressure_hpa": pressure.round(1), "wind_speed_kmh": wind.round(1),
        "flood_label": label, "data_source": "synthetic_supplementary",
    })
    drop_idx = r.choice(N_DAYS_SYNTH, size=int(N_DAYS_SYNTH * 0.02), replace=False)
    df.loc[drop_idx, ["rainfall_mm_24h", "river_level_m"]] = np.nan
    gap_start = r.integers(50, N_DAYS_SYNTH - 60)
    df.loc[gap_start:gap_start + 2, "soil_moisture"] = np.nan
    return df

In [ ]:
synth_region_ids = [rid for rid, _, _ in REGIONS if rid not in REAL_REGIONS]
synth_frames = [simulate_synthetic_region(rid, seed=SEED + i) for i, rid in enumerate(synth_region_ids)]

real_combined = pd.concat(real_frames, ignore_index=True).drop(columns=["centroid_lat", "centroid_lon"])
synth_combined = pd.concat(synth_frames, ignore_index=True)
stations = pd.concat([real_combined, synth_combined], ignore_index=True)
print(stations.groupby(["region_id", "data_source"]).agg(days=("date", "count"), positives=("flood_label", "sum")))
stations.to_csv(RAW / "station_readings.csv", index=False)

In [ ]:
# SANLC-style static land cover composition per region — still synthetic for all 5 regions,
# no imagery/GIS pipeline exists yet to derive this from real SANLC data. See DATA_SOURCES.md.
land_cover = []
for rid, name, prov in REGIONS:
    r = np.random.default_rng(hash(rid) % (2**32))
    coastal = 1.0 if "ETHEKWINI" in rid or "KINGCETSHWAYO" in rid else 0.3
    urban = r.uniform(0.15, 0.45) * (1.3 if "ETHEKWINI" in rid else 1.0)
    water = r.uniform(0.02, 0.06) * (1.5 if coastal else 1.0)
    veg = r.uniform(0.3, 0.55)
    bare = max(0.0, 1 - urban - water - veg)
    total = urban + water + veg + bare
    land_cover.append({"region_id": rid, "name": name, "province": prov,
                        "urban_pct": urban / total, "water_pct": water / total,
                        "vegetation_pct": veg / total, "bare_pct": bare / total})
land_cover_df = pd.DataFrame(land_cover)
land_cover_df.to_csv(RAW / "sanlc_land_cover.csv", index=False)
land_cover_df

In [ ]:
# Historical event log. Real regions: derived from the label column above, tagged by source
# (known-public-reporting anchor vs heuristic threshold). Synthetic regions: derived from
# their injected label episodes, as before.
events = []
for rid, df in stations.groupby("region_id"):
    df = df.sort_values("date").reset_index(drop=True)
    lbl = df["flood_label"].values
    source = df["data_source"].iloc[0]
    in_event, start = False, None
    for i, v in enumerate(lbl):
        if v and not in_event:
            in_event, start = True, df["date"].iloc[i]
        if in_event and (not v or i == len(lbl) - 1):
            end = df["date"].iloc[i - 1 if not v else i]
            label_source = ("public_reporting_anchor"
                             if source == "real_openmeteo" and str(start.date()) in
                             KNOWN_FLOOD_DATES.get(rid, []) else
                             "heuristic_threshold" if source == "real_openmeteo" else
                             "synthetic_injected")
            events.append({"region_id": rid, "event_type": "flood", "start_date": start,
                           "end_date": end, "label_source": label_source})
            in_event = False
events_df = pd.DataFrame(events)
events_df.to_csv(RAW / "historical_events.csv", index=False)
events_df

In [ ]:
# Sentinel-2-style multispectral tiles — synthetic for all 5 regions (no imagery pipeline yet).
def make_tiles(label: np.ndarray, seed: int) -> np.ndarray:
    r = np.random.default_rng(seed)
    tiles = r.normal(0.2, 0.05, (len(label), TILE, TILE, 4)).astype("float32")
    yy, xx = np.mgrid[0:TILE, 0:TILE]
    cy, cx = TILE / 2, TILE / 2
    for i, lab in enumerate(label):
        if lab:
            radius = r.uniform(3, 7)
            blob = (((yy - cy) ** 2 + (xx - cx) ** 2) < radius ** 2).astype("float32")
            tiles[i, :, :, 3] += blob * r.uniform(0.4, 0.8)
    return np.clip(tiles, 0, 1)

tile_arrays = {}
for i, (rid, df) in enumerate(stations.groupby("region_id")):
    df = df.sort_values("date")
    tile_arrays[rid] = make_tiles(df["flood_label"].values, SEED + i)
np.savez_compressed(RAW / "satellite_tiles.npz", **tile_arrays)
{k: v.shape for k, v in tile_arrays.items()}

In [ ]:
def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

all_files = list(RAW.glob("*")) + list(SRC.glob("*.xlsx"))
checksums = {str(p.relative_to(RAW.parent)): sha256_of(p) for p in all_files if p.is_file()}
(RAW / "CHECKSUMS.sha256.json").write_text(json.dumps(checksums, indent=2))
(RAW / "DATASET_VERSION.txt").write_text(DATASET_VERSION)
checksums